In [1]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms

import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

lib_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Code'
os.chdir(lib_dir)
print('library directory: ' + lib_dir)
from modules.networks_clf import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *
from modules.train import net_train_AnyNet_L, net_train_ViT_L, net_train_RNN_L, net_train_LC_L
from modules.metrics import accu_fun

library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code
Library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code


In [2]:
################################################################################
import subprocess
import sys
required = {'ONE-api', 'brain', 'ibllib'}
subprocess.check_call([sys.executable, '-m', 'pip', 'install', *required])

from one.api import ONE
# from brainbox.io.one import SessionLoader, SpikeSortingLoader
# from ibllib.atlas import AllenAtlas
# from brainbox.io.spikeglx import Streamer
# from neurodsp.voltage import destripe
# from datetime import datetime
# from pprint import pprint

# ba = AllenAtlas()
# br = ba.regions
# ba.compute_regions_volume()


In [3]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-58de84bd-57a2-a136-698b-191cbaf4bbc1)


In [4]:
# !pip install captum
# from captum.attr import IntegratedGradients
# from captum.attr import LayerConductance
# from captum.attr import NeuronConductance

# from captum.attr import Saliency
# from captum.attr import DeepLift
# from captum.attr import NoiseTunnel
# from captum.attr import visualization as viz

In [5]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [6]:
train_args = {
    'overfitting_thres':0.60,
    'lr':5e-4,
    'norm':True,
    'temp':[True, True],
    # 'epochs':50,
    # 'save_dir':'/content/drive/MyDrive/Project/BrainRegionId/Project43',
}


In [7]:
# @title Load data

file_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/'
brain_signal_lfp = torch.load(file_dir + '/brain_signal_lfp1.pt')
for file_id in [2, 3, 4, 5]:
    brain_signal_lfp = torch.concat([brain_signal_lfp, torch.load(file_dir + f'/brain_signal_lfp{file_id}.pt')], dim=0)
    print(f'Load file id: {file_id}')



Load file id: 2
Load file id: 3
Load file id: 4
Load file id: 5


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([scalar])` or the `torch.serialization.safe_globals([scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weigh

In [8]:
list_dict = torch.load(file_dir + '/list_dict.pt', weights_only=False)

brain_region_index = list_dict['brain_region_index']
brain_region_index_Cosmos = list_dict['brain_region_index_Cosmos']
coordinate_list = list_dict['coordinate_list']
acronym_selec_list = list_dict['acronym_selec_list']

In [9]:
if len(brain_signal_lfp) == len(brain_region_index):
    print('Matched, no damage!')

Matched, no damage!


In [ ]:
# acronym_interp = ['VISp1', 'VISp2/3', 'VISp4', 'VISp5', 'VISp6a', 'VISp6b']

In [14]:
ind = 0

key = 'stimOn_times'

model_dir = '/content/drive/MyDrive/Project/BrainRegionId/Science/Model/Allen'

if device.type != 'cuda':

    Classifier_LCC = torch.load(model_dir + f'/LC_L_Allen_chance_{key}_{ind}.pth', map_location=torch.device('cpu'))
    Classifier_LC = torch.load(model_dir + f'/LC_L_Allen_{key}_{ind}.pth', map_location=torch.device('cpu'))


    Classifier_AnyNet = torch.load(model_dir + f'/AnyNet_L_Allen_{key}_{ind}.pth', map_location=torch.device('cpu'))
    Classifier_ViT = torch.load(model_dir + f'/ViT_L_Allen_{key}_{ind}.pth', map_location=torch.device('cpu'))
    Classifier_RNN = torch.load(model_dir + f'/RNN_L_Allen_{key}_{ind}.pth', map_location=torch.device('cpu'))

elif device.type == 'cuda':

    Classifier_LCC = torch.load(model_dir + f'/LC_L_Allen_chance_{key}_{ind}.pth', weights_only=False).to(device)
    Classifier_LC = torch.load(model_dir + f'/LC_L_Allen_{key}_{ind}.pth', weights_only=False).to(device)


    Classifier_AnyNet = torch.load(model_dir + f'/AnyNet_L_Allen_{key}_{ind}.pth', weights_only=False).to(device)
    Classifier_ViT = torch.load(model_dir + f'/ViT_L_Allen_{key}_{ind}.pth', weights_only=False).to(device)
    Classifier_RNN = torch.load(model_dir + f'/RNN_L_Allen_{key}_{ind}.pth', weights_only=False).to(device)

subject_od_ind = torch.load(model_dir + f'/subject_od_ind_Allen_{key}{0}.pt', weights_only=False)
subject_od_list = torch.load(model_dir + f'/subject_od_list_Allen_{key}{0}.pt', weights_only=False)


In [15]:
train_ind, valid_ind, test_ind, test_subject_ind = dat_ind_gen(list_dict, subject_od_ind, key)

In [17]:
sample_num = 128
print(sample_num)
Mat_dict = {}
Classifier_name = [ 'AnyNet', 'ViT', 'RNN']
Classifier_list = [Classifier_AnyNet, Classifier_ViT, Classifier_RNN]
for Classifier_ii, Classifier in enumerate(Classifier_list):
    confusion_mat = np.zeros((len(acronym_list), len(acronym_list)))
    Classifier.eval()
    acu_test_indiv = []
    acronym_test = []
    for acronym_ii, acronym in enumerate(acronym_list):
        test_ind_acronym_ii = np.intersect1d(np.argwhere(np.array(list_dict['brain_region_atlas']) == acronym).flatten(), test_ind)
        if len(test_ind_acronym_ii) < 1:
            continue
        elif len(test_ind_acronym_ii) < sample_num:
            test_indiv = test_ind_acronym_ii
        elif len(test_ind_acronym_ii) >= sample_num:
            test_indiv = test_ind_acronym_ii[np.random.choice(len(test_ind_acronym_ii), sample_num, replace=False)]
        x_test1 = brain_signal_lfp[test_indiv,:]
        y_test = brain_region_index[test_indiv]
        x_test = lfp_spectro(x_test1, spectro_args, train_args)

        if Classifier_name[Classifier_ii] == 'RNN':
            py_test = Classifier(x_test.to(device).squeeze(1).permute(0, 2, 1))
        elif Classifier_name[Classifier_ii] in ['Chance', 'Linear']:
            py_test = Classifier(x_test.to(device).squeeze(1).flatten(start_dim=1))
        else:
            py_test = Classifier(x_test.to(device))
        p, edg = torch.histogram(torch.argmax(py_test.cpu(), dim=1).to(torch.float32), bins=len(acronym_list), range=(0, len(acronym_list)), density=True)
        confusion_mat[acronym_ii, :] = p.detach().cpu().numpy()
        print(acronym_ii)
    Mat_dict[Classifier_name[Classifier_ii]] = confusion_mat

128
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
68
69
73
76
77
78
80
81
82
83
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
201
202
203
204
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
278
279
280
282
283
284
285
286


In [18]:
Mat_dict

{'AnyNet': array([[0.25      , 0.55000001, 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.03125   , 0.5390625 , 0.015625  , ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.56363636, ..., 0.        , 0.        ,
         0.        ],
        ...,
        [0.        , 0.        , 0.        , ..., 0.4453125 , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.015625  , 0.515625  ,
         0.015625  ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.03125   ,
         0.4765625 ]]),
 'ViT': array([[0.1       , 0.25      , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.0234375 , 0.421875  , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.37272727, ..., 0.        , 0.        ,
         0.        ],
        ...,
        [0.        , 0.        , 0.        , ..., 0.4140625 , 0.015625  ,
         0.      

In [19]:
torch.save(Mat_dict, f'/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/Confusion_mat_{key}.pt')